# System prompts y parámetros en local

Hasta ahora habia usado system prompts con la API de Anthropic.
Aquí hacemos lo mismo con Ollama — mismo concepto, modelo local, sin coste por token.

La ventaja: podemos experimentar con temperatura y otros parámetros
libremente sin preocuparnos del gasto.

## Qué veremos
- Cómo el system prompt cambia el comportamiento del modelo
- Qué efecto tiene la temperatura en las respuestas
- Diferencia entre un modelo "sin personalidad" y uno bien configurado

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:7b"

def chat(user_msg: str, system: str = "", temperature: float = 0.7) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})

    response = httpx.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "options": {"temperature": temperature}
        },
        timeout=60
    )
    return response.json()["message"]["content"]

## El system prompt cambia todo

Sin system prompt el modelo responde con su personalidad por defecto.
Con system prompt le damos un rol, restricciones y contexto permanente.
Mismo input, respuesta completamente diferente.

In [ ]:
pregunta = "¿Cómo soluciono un error 500 en mi API?"

# Sin system prompt
sin_sistema = chat(pregunta)
print("SIN SYSTEM PROMPT:")
print(sin_sistema)
print("\n" + "="*50 + "\n")

# Con system prompt
con_sistema = chat(
    pregunta,
    system="""Eres un senior backend engineer especializado en APIs REST.
Respondes de forma concisa y técnica, siempre en bullet points.
Nunca das explicaciones largas — solo pasos accionables."""
)
print("CON SYSTEM PROMPT:")
print(con_sistema)

## Temperatura — aleatoriedad controlada

Controla cuánta variedad introduce el modelo al generar texto.

- `temperature=0` → siempre la misma respuesta (determinista)
- `temperature=0.2` → casi determinista, ligera variedad
- `temperature=0.7` → creativo y variable, útil para brainstorming

Ejecutamos el mismo prompt tres veces con temperature=0
para demostrar el determinismo, y tres veces con temperature=0.7
para ver la variedad.

In [ ]:
prompt = "Dame un nombre creativo para una startup de IA"
system = "Eres un experto en branding tecnológico."

print("=== TEMPERATURE 0 (determinista) ===")
for i in range(3):
    r = chat(prompt, system=system, temperature=0)
    print(f"[{i+1}] {r}\n")

print("=== TEMPERATURE 0.7 (creativo) ===")
for i in range(3):
    r = chat(prompt, system=system, temperature=0.7)
    print(f"[{i+1}] {r}\n")

## Conversación multi-turno — memoria entre mensajes

Hasta ahora cada llamada es independiente: el modelo no recuerda
lo que dijiste antes. Para mantener contexto hay que pasar
el historial completo de mensajes en cada llamada.

Es exactamente lo que hace el chat de Claude internamente —
no hay magia, solo acumulación del historial.

In [ ]:
def chat_con_memoria(system: str = ""):
    historial = []
    if system:
        historial.append({"role": "system", "content": system})

    print("Chat iniciado (escribe 'salir' para terminar)\n")

    while True:
        user_input = input("Tú: ")
        if user_input.lower() == "salir":
            break

        historial.append({"role": "user", "content": user_input})

        response = httpx.post(
            OLLAMA_URL,
            json={"model": MODEL, "messages": historial, "stream": False},
            timeout=60
        )

        assistant_msg = response.json()["message"]["content"]
        historial.append({"role": "assistant", "content": assistant_msg})

        print(f"\nModelo: {assistant_msg}\n")

    return historial

historial = chat_con_memoria(system="Eres un asistente técnico conciso.")